# LunarLander PPO experiments

This notebook explores curriculum learning in `LunarLander-v3` and expands on it to teach a model to perform a full 360 rotation before landing safely.

A single PPO policy is trained progressively across four reward
configurations:

1. discover and complete a full rotation;
2. recover upright and attempt a landing;
3. reach the landing zone;
4. control the final descent and land softly.

At the end of each phase, locally saved checkpoints are evaluated on
the same deterministic seeds. The checkpoint that best matches that
phase's task-specific criteria is selected as the starting point for
the next phase.

## 1. Runtime setup

### 1.1 Install dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y -qq swig cmake ffmpeg xvfb libgl1

%pip install -q \
    "stable-baselines3[extra]==2.9.0" \
    "gymnasium[box2d]==1.3.0" \
    "huggingface-hub<2" \
    pyvirtualdisplay

### 1.2 Imports and virtual display

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Iterable, Sequence
import json
import os
import shutil
import textwrap

import gymnasium as gym
import numpy as np
import pandas as pd

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    notebook_login,
    ModelCard,
    ModelCardData,
)
from huggingface_hub.utils import (
    EntryNotFoundError,
    RepositoryNotFoundError,
)

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    EvalCallback,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

from pyvirtualdisplay import Display
from IPython.display import Video, display


if not os.environ.get("DISPLAY"):
    virtual_display = Display(
        visible=False,
        size=(1400, 900),
    )
    virtual_display.start()

    print(
        "Virtual display started:",
        virtual_display.is_alive(),
    )
else:
    print("Using existing display:", os.environ["DISPLAY"])

### 1.3 Global constants

In [ ]:
# Global constants
HF_REPO_ID = (
    "KaptainKris/"
    "ppo-LunarLander-v3-flip"
)

OUTPUT_ROOT = Path(
    "/content/rl-experiments"
)

EVALUATION_START_SEED = 20_000

## 2. Custom environment

The plan for this model is to have the same architecture as before, but to change the environment's reward function to reward doing a full 360 degree flip before landing safely.

### 2.1 Reward wrapper

The wrapper modifies the reward while preserving the original
`LunarLander-v3` dynamics.

It also extends the observation from 8 to 11 values by appending:

- directed rotation progress;
- whether the flip has been completed;
- whether post-flip recovery has been completed.

The wrapper is written to a standalone Python file so the exact
environment definition can be uploaded alongside the trained model.

In [ ]:
%%writefile /content/flip_landing_reward_wrapper_v2.py

from __future__ import annotations

from collections.abc import Mapping
import math

import gymnasium as gym
import numpy as np



DEFAULT_REWARD_CONFIG = {
    # ------------------------------------------------------------------
    # 1. Task-stage definitions
    #
    # These values define when the wrapper considers the flip and
    # post-flip recovery stages complete.
    # ------------------------------------------------------------------

    "required_rotations": 1.0,
    "rotation_direction": 1,
    "upright_tolerance_radians": 0.30,
    "recovery_angular_velocity_tolerance": 0.50,

    # ------------------------------------------------------------------
    # 2. Original LunarLander reward contribution
    #
    # The custom reward is added to a weighted version of Gymnasium's
    # original LunarLander reward.
    # ------------------------------------------------------------------

    "pre_flip_original_reward_weight": 0.25,
    "post_flip_original_reward_weight": 1.0,

    # ------------------------------------------------------------------
    # 3. Positive stage and terminal rewards
    #
    # Progress is paid only when the maximum achieved rotation progress
    # increases. The other rewards are one-off stage rewards.
    # ------------------------------------------------------------------

    "rotation_progress_bonus": 150.0,
    "flip_completion_bonus": 200.0,
    "recovery_bonus": 100.0,
    "flip_landing_bonus": 750.0,

    # ------------------------------------------------------------------
    # 4. Terminal failure penalties
    # ------------------------------------------------------------------

    "landing_without_flip_penalty": 300.0,
    "no_flip_terminal_penalty": 300.0,
    "failed_landing_penalty": 150.0,
    "outside_zone_landing_penalty": 1_000.0,
    "in_zone_crash_penalty": 1_200.0,

    # ------------------------------------------------------------------
    # 5. Potential-difference shaping controls
    #
    # These control how changes in post-flip state quality are converted
    # into a dense per-step reward.
    # ------------------------------------------------------------------

    "post_flip_shaping_weight": 1.0,
    "post_flip_shaping_gamma": 0.995,
    "post_flip_shaping_clip": 10.0,

    # ------------------------------------------------------------------
    # 6. Post-flip state-quality weights
    #
    # These determine the relative importance of position, velocity,
    # attitude, angular velocity and leg contact in the potential.
    # ------------------------------------------------------------------

    "post_flip_center_weight": 50.0,
    "post_flip_horizontal_speed_weight": 30.0,
    "post_flip_vertical_speed_weight": 30.0,
    "post_flip_angle_weight": 20.0,
    "post_flip_angular_speed_weight": 10.0,
    "post_flip_leg_contact_weight": 10.0,

    # ------------------------------------------------------------------
    # 7. Horizontal landing-zone guidance
    # ------------------------------------------------------------------

    "landing_zone_half_width": 0.20,
    "post_flip_zone_excess_weight": 200.0,
    "post_flip_target_vx_gain": 0.50,
    "post_flip_max_target_vx": 0.35,
    "post_flip_horizontal_deadband": 0.08,

    # ------------------------------------------------------------------
    # 8. Vertical descent and soft-touchdown guidance
    # ------------------------------------------------------------------

    "post_flip_target_vy_high": -0.45,
    "post_flip_target_vy_near_ground": -0.12,
    "near_ground_height": 0.60,
    "safe_touchdown_vertical_speed": 0.18,
    "near_ground_overspeed_weight": 120.0,
}


class FlipLandingRewardWrapper(gym.Wrapper):
    """
    Stage-aware LunarLander objective:

    1. Complete one full rotation in the selected direction.
    2. Arrest the spin and recover to an upright attitude.
    3. Re-centre, stabilise and land safely.

    The original eight-value observation is extended with:
    - signed rotation progress in [-1, 1];
    - a flip-completed flag;
    - a post-flip-recovery-completed flag.

    Post-flip recovery uses a potential-difference reward based on
    horizontal position, horizontal speed, attitude, angular speed,
    and leg contact. This rewards improvement rather than repeatedly
    paying the agent simply for occupying a favourable state.
    """

        def __init__(
        self,
        env: gym.Env,
        *,
        reward_config: (
            Mapping[str, float | int]
            | None
        ) = None,
    ):
        """
        Initialise the reward wrapper.

        DEFAULT_REWARD_CONFIG is the single source of default values.
        `reward_config` may override any subset of those values for a
        particular curriculum phase.
        """

        super().__init__(env)

        supplied_config = dict(
            reward_config or {}
        )

        # Reject misspelled or obsolete parameter names. Without this
        # check, an accidental typo could silently leave the default
        # value active.
        unknown_parameters = (
            set(supplied_config)
            - set(DEFAULT_REWARD_CONFIG)
        )

        if unknown_parameters:
            raise ValueError(
                "Unknown reward parameters: "
                f"{sorted(unknown_parameters)}"
            )

        # Make a new dictionary so the global defaults are never mutated.
        config = {
            **DEFAULT_REWARD_CONFIG,
            **supplied_config,
        }

        # ==============================================================
        # 1. Task-stage definitions
        # ==============================================================

        self.required_rotations = float(
            config["required_rotations"]
        )

        self.rotation_direction = int(
            config["rotation_direction"]
        )

        self.upright_tolerance_radians = float(
            config[
                "upright_tolerance_radians"
            ]
        )

        self.recovery_angular_velocity_tolerance = float(
            config[
                "recovery_angular_velocity_tolerance"
            ]
        )

        # ==============================================================
        # 2. Original LunarLander reward contribution
        # ==============================================================

        self.pre_flip_original_reward_weight = float(
            config[
                "pre_flip_original_reward_weight"
            ]
        )

        self.post_flip_original_reward_weight = float(
            config[
                "post_flip_original_reward_weight"
            ]
        )

        # ==============================================================
        # 3. Positive stage and terminal rewards
        # ==============================================================

        self.rotation_progress_bonus = float(
            config[
                "rotation_progress_bonus"
            ]
        )

        self.flip_completion_bonus = float(
            config[
                "flip_completion_bonus"
            ]
        )

        self.recovery_bonus = float(
            config["recovery_bonus"]
        )

        self.flip_landing_bonus = float(
            config["flip_landing_bonus"]
        )

        # ==============================================================
        # 4. Terminal failure penalties
        # ==============================================================

        self.landing_without_flip_penalty = float(
            config[
                "landing_without_flip_penalty"
            ]
        )

        self.no_flip_terminal_penalty = float(
            config[
                "no_flip_terminal_penalty"
            ]
        )

        self.failed_landing_penalty = float(
            config[
                "failed_landing_penalty"
            ]
        )

        self.outside_zone_landing_penalty = float(
            config[
                "outside_zone_landing_penalty"
            ]
        )

        self.in_zone_crash_penalty = float(
            config[
                "in_zone_crash_penalty"
            ]
        )

        # ==============================================================
        # 5. Potential-difference shaping controls
        # ==============================================================

        self.post_flip_shaping_weight = float(
            config[
                "post_flip_shaping_weight"
            ]
        )

        self.post_flip_shaping_gamma = float(
            config[
                "post_flip_shaping_gamma"
            ]
        )

        self.post_flip_shaping_clip = float(
            config[
                "post_flip_shaping_clip"
            ]
        )

        # ==============================================================
        # 6. Post-flip state-quality weights
        # ==============================================================

        self.post_flip_center_weight = float(
            config[
                "post_flip_center_weight"
            ]
        )

        self.post_flip_horizontal_speed_weight = float(
            config[
                "post_flip_horizontal_speed_weight"
            ]
        )

        self.post_flip_vertical_speed_weight = float(
            config[
                "post_flip_vertical_speed_weight"
            ]
        )

        self.post_flip_angle_weight = float(
            config[
                "post_flip_angle_weight"
            ]
        )

        self.post_flip_angular_speed_weight = float(
            config[
                "post_flip_angular_speed_weight"
            ]
        )

        self.post_flip_leg_contact_weight = float(
            config[
                "post_flip_leg_contact_weight"
            ]
        )

        # ==============================================================
        # 7. Horizontal landing-zone guidance
        # ==============================================================

        self.landing_zone_half_width = float(
            config[
                "landing_zone_half_width"
            ]
        )

        self.post_flip_zone_excess_weight = float(
            config[
                "post_flip_zone_excess_weight"
            ]
        )

        self.post_flip_target_vx_gain = float(
            config[
                "post_flip_target_vx_gain"
            ]
        )

        self.post_flip_max_target_vx = float(
            config[
                "post_flip_max_target_vx"
            ]
        )

        self.post_flip_horizontal_deadband = float(
            config[
                "post_flip_horizontal_deadband"
            ]
        )

        # ==============================================================
        # 8. Vertical descent and soft-touchdown guidance
        # ==============================================================

        self.post_flip_target_vy_high = float(
            config[
                "post_flip_target_vy_high"
            ]
        )

        self.post_flip_target_vy_near_ground = float(
            config[
                "post_flip_target_vy_near_ground"
            ]
        )

        self.near_ground_height = float(
            config[
                "near_ground_height"
            ]
        )

        self.safe_touchdown_vertical_speed = float(
            config[
                "safe_touchdown_vertical_speed"
            ]
        )

        self.near_ground_overspeed_weight = float(
            config[
                "near_ground_overspeed_weight"
            ]
        )

        # ==============================================================
        # Configuration validation
        # ==============================================================

        if self.required_rotations <= 0.0:
            raise ValueError(
                "required_rotations must be positive."
            )

        if self.rotation_direction not in (
            -1,
            1,
        ):
            raise ValueError(
                "rotation_direction must be "
                "either -1 or 1."
            )

        if not (
            0.0
            < self.upright_tolerance_radians
            <= math.pi
        ):
            raise ValueError(
                "upright_tolerance_radians must "
                "be in the interval (0, pi]."
            )

        if (
            self.recovery_angular_velocity_tolerance
            < 0.0
        ):
            raise ValueError(
                "recovery_angular_velocity_tolerance "
                "must not be negative."
            )

        if not (
            0.0
            <= self.post_flip_shaping_gamma
            <= 1.0
        ):
            raise ValueError(
                "post_flip_shaping_gamma must "
                "be in [0, 1]."
            )

        if self.post_flip_shaping_clip <= 0.0:
            raise ValueError(
                "post_flip_shaping_clip must "
                "be positive."
            )

        if not (
            0.0
            < self.landing_zone_half_width
            < 1.0
        ):
            raise ValueError(
                "landing_zone_half_width must "
                "be between 0 and 1."
            )

        if self.post_flip_max_target_vx <= 0.0:
            raise ValueError(
                "post_flip_max_target_vx must "
                "be positive."
            )

        if self.near_ground_height <= 0.0:
            raise ValueError(
                "near_ground_height must be "
                "positive."
            )

        # All parameters in this collection represent rewards,
        # penalties, weights, tolerances or magnitudes. Negative values
        # would invert their intended meaning.
        non_negative_parameters = {
            "pre_flip_original_reward_weight": (
                self.pre_flip_original_reward_weight
            ),
            "post_flip_original_reward_weight": (
                self.post_flip_original_reward_weight
            ),
            "rotation_progress_bonus": (
                self.rotation_progress_bonus
            ),
            "flip_completion_bonus": (
                self.flip_completion_bonus
            ),
            "recovery_bonus": (
                self.recovery_bonus
            ),
            "flip_landing_bonus": (
                self.flip_landing_bonus
            ),
            "landing_without_flip_penalty": (
                self.landing_without_flip_penalty
            ),
            "no_flip_terminal_penalty": (
                self.no_flip_terminal_penalty
            ),
            "failed_landing_penalty": (
                self.failed_landing_penalty
            ),
            "outside_zone_landing_penalty": (
                self.outside_zone_landing_penalty
            ),
            "in_zone_crash_penalty": (
                self.in_zone_crash_penalty
            ),
            "post_flip_shaping_weight": (
                self.post_flip_shaping_weight
            ),
            "post_flip_center_weight": (
                self.post_flip_center_weight
            ),
            "post_flip_horizontal_speed_weight": (
                self.post_flip_horizontal_speed_weight
            ),
            "post_flip_vertical_speed_weight": (
                self.post_flip_vertical_speed_weight
            ),
            "post_flip_angle_weight": (
                self.post_flip_angle_weight
            ),
            "post_flip_angular_speed_weight": (
                self.post_flip_angular_speed_weight
            ),
            "post_flip_leg_contact_weight": (
                self.post_flip_leg_contact_weight
            ),
            "post_flip_zone_excess_weight": (
                self.post_flip_zone_excess_weight
            ),
            "post_flip_target_vx_gain": (
                self.post_flip_target_vx_gain
            ),
            "post_flip_horizontal_deadband": (
                self.post_flip_horizontal_deadband
            ),
            "safe_touchdown_vertical_speed": (
                self.safe_touchdown_vertical_speed
            ),
            "near_ground_overspeed_weight": (
                self.near_ground_overspeed_weight
            ),
        }

        for (
            parameter_name,
            parameter_value,
        ) in non_negative_parameters.items():
            if parameter_value < 0.0:
                raise ValueError(
                    f"{parameter_name} must not "
                    "be negative."
                )

        # Downward velocity is represented by a negative value.
        if self.post_flip_target_vy_high > 0.0:
            raise ValueError(
                "post_flip_target_vy_high must "
                "be zero or negative."
            )

        if (
            self.post_flip_target_vy_near_ground
            > 0.0
        ):
            raise ValueError(
                "post_flip_target_vy_near_ground "
                "must be zero or negative."
            )

        # For example, -0.12 is slower than -0.45.
        if (
            self.post_flip_target_vy_near_ground
            < self.post_flip_target_vy_high
        ):
            raise ValueError(
                "The near-ground target vertical "
                "speed must be slower than the "
                "high-altitude target."
            )

        if (
            self.post_flip_horizontal_deadband
            > self.landing_zone_half_width
        ):
            raise ValueError(
                "post_flip_horizontal_deadband "
                "must not exceed "
                "landing_zone_half_width."
            )

        # Convert the configured number of rotations to radians.
        self.required_rotation = (
            2.0
            * math.pi
            * self.required_rotations
        )

        # ==============================================================
        # Per-episode state
        #
        # These values are reset at the start of every episode. They are
        # not reward hyperparameters.
        # ==============================================================

        self.previous_angle = 0.0
        self.cumulative_rotation = 0.0
        self.maximum_rotation_progress = 0.0
        self.flip_completed = False
        self.recovery_completed = False

        self.previous_post_flip_potential: (
            float | None
        ) = None

        # ==============================================================
        # Extended observation space
        #
        # Append:
        # 1. signed rotation progress;
        # 2. flip-completed flag;
        # 3. recovery-completed flag.
        # ==============================================================

        base_space = self.env.observation_space

        if not isinstance(
            base_space,
            gym.spaces.Box,
        ):
            raise TypeError(
                "The wrapped environment must "
                "have a Box observation space."
            )

        extra_low = np.asarray(
            [
                -1.0,
                0.0,
                0.0,
            ],
            dtype=np.float32,
        )

        extra_high = np.asarray(
            [
                1.0,
                1.0,
                1.0,
            ],
            dtype=np.float32,
        )

        self.observation_space = (
            gym.spaces.Box(
                low=np.concatenate(
                    [
                        base_space.low.astype(
                            np.float32
                        ),
                        extra_low,
                    ]
                ),
                high=np.concatenate(
                    [
                        base_space.high.astype(
                            np.float32
                        ),
                        extra_high,
                    ]
                ),
                dtype=np.float32,
            )
        )

    @staticmethod
    def _wrap_angle(angle: float) -> float:
        """Wrap an angle to [-pi, pi]."""

        return float(
            np.arctan2(
                np.sin(angle),
                np.cos(angle),
            )
        )

    def _signed_rotation_progress(self) -> float:
        return float(
            np.clip(
                self.cumulative_rotation
                / self.required_rotation,
                -1.0,
                1.0,
            )
        )

    def _augment_observation(
        self,
        observation: np.ndarray,
    ) -> np.ndarray:
        return np.concatenate(
            [
                np.asarray(
                    observation,
                    dtype=np.float32,
                ),
                np.asarray(
                    [
                        self._signed_rotation_progress(),
                        float(self.flip_completed),
                        float(self.recovery_completed),
                    ],
                    dtype=np.float32,
                ),
            ]
        )

    def _post_flip_potential(
        self,
        observation: np.ndarray,
    ) -> float:
        """
        Higher values represent safer post-flip states.

        LunarLander-v3 observation positions:
        0 x position, 2 x velocity, 4 angle, 5 angular velocity,
        6 left-leg contact and 7 right-leg contact.
        """

        x_position = float(
            observation[0]
        )

        y_position = float(
            observation[1]
        )

        horizontal_speed = float(
            observation[2]
        )

        vertical_speed = float(
            observation[3]
        )

        height_ratio = float(
            np.clip(
                y_position
                / self.near_ground_height,
                0.0,
                1.0,
            )
        )

        target_vertical_speed = (
            self.post_flip_target_vy_near_ground
            + height_ratio
            * (
                self.post_flip_target_vy_high
                - self.post_flip_target_vy_near_ground
            )
        )

        vertical_speed_error = abs(
            vertical_speed
            - target_vertical_speed
        )

        wrapped_angle = abs(
            self._wrap_angle(
                float(observation[4])
            )
        )

        angular_speed = abs(
            float(observation[5])
        )

        leg_contacts = (
            float(observation[6] > 0.5)
            + float(observation[7] > 0.5)
        )

        # Distance beyond the two landing-zone flags.
        zone_excess = max(
            abs(x_position)
            - self.landing_zone_half_width,
            0.0,
        )

        # When left of the pad, target positive horizontal
        # velocity. When right of the pad, target negative
        # horizontal velocity. Close to the centre, target
        # velocity approaches zero.
        horizontal_guidance_factor = float(
            np.clip(
                y_position / 0.50,
                0.0,
                1.0,
            )
        )

        horizontal_error = max(
            abs(x_position)
            - self.post_flip_horizontal_deadband,
            0.0,
        )

        target_horizontal_speed = float(
            np.clip(
                -self.post_flip_target_vx_gain
                * x_position
                * horizontal_guidance_factor,
                -self.post_flip_max_target_vx,
                self.post_flip_max_target_vx,
            )
        )

        horizontal_speed_error = abs(
            horizontal_speed
            - target_horizontal_speed
        )

        # Increase the importance of entering the landing
        # zone as the lander approaches the ground.
        near_ground_factor = float(
            np.clip(
                1.0
                - max(y_position, 0.0)
                / 0.75,
                0.0,
                1.0,
            )
        )

        effective_zone_weight = (
            self.post_flip_zone_excess_weight
            * (1.0 + near_ground_factor)
        )

        return float(
            -self.post_flip_center_weight
            * horizontal_error

            -effective_zone_weight
            * zone_excess

            -self.post_flip_horizontal_speed_weight
            * horizontal_speed_error

            -self.post_flip_vertical_speed_weight
            * vertical_speed_error

            -self.post_flip_angle_weight
            * wrapped_angle

            -self.post_flip_angular_speed_weight
            * angular_speed

            +self.post_flip_leg_contact_weight
            * leg_contacts
        )

    def reset(
        self,
        *,
        seed: int | None = None,
        options: dict | None = None,
    ):
        observation, info = self.env.reset(
            seed=seed,
            options=options,
        )

        self.previous_angle = float(observation[4])
        self.cumulative_rotation = 0.0
        self.maximum_rotation_progress = 0.0
        self.flip_completed = False
        self.recovery_completed = False
        self.previous_post_flip_potential = None

        info = dict(info)
        info.update(
            {
                "flip_completed": False,
                "recovery_completed": False,
                "rotation_progress": 0.0,
                "rotations_completed": 0.0,
                "rotation_progress_reward": 0.0,
                "flip_completion_reward": 0.0,
                "recovery_reward": 0.0,
                "post_flip_shaping_reward": 0.0,
                "flip_landing_bonus": 0.0,
                "terminal_adjustment": 0.0,
            }
        )

        return self._augment_observation(observation), info

    def step(self, action):
        # 1. Advance the original LunarLander environment.
        (
            observation,
            original_reward,
            terminated,
            truncated,
            info,
        ) = self.env.step(action)

        # 2. Accumulate directed angular movement and reward new flip progress.
        current_angle = float(observation[4])
        angular_velocity = float(observation[5])

        angle_change = self._wrap_angle(
            current_angle - self.previous_angle
        )
        self.previous_angle = current_angle

        directed_change = (
            self.rotation_direction * angle_change
        )
        self.cumulative_rotation += directed_change

        rotation_progress = float(
            np.clip(
                self.cumulative_rotation
                / self.required_rotation,
                0.0,
                1.0,
            )
        )

        new_progress = max(
            0.0,
            rotation_progress
            - self.maximum_rotation_progress,
        )
        progress_reward = (
            self.rotation_progress_bonus
            * new_progress
        )
        self.maximum_rotation_progress = max(
            self.maximum_rotation_progress,
            rotation_progress,
        )

        completion_reward = 0.0

        # 3. Detect the first completed full rotation.
        if (
            rotation_progress >= 1.0
            and not self.flip_completed
        ):
            self.flip_completed = True
            completion_reward = (
                self.flip_completion_bonus
            )

        # 4. Detect upright post-flip recovery.
        wrapped_angle = abs(
            self._wrap_angle(current_angle)
        )
        upright = (
            wrapped_angle
            <= self.upright_tolerance_radians
        )

        recovery_reward = 0.0

        if (
            self.flip_completed
            and not self.recovery_completed
            and upright
            and abs(angular_velocity)
            <= self.recovery_angular_velocity_tolerance
        ):
            self.recovery_completed = True
            recovery_reward = self.recovery_bonus


        post_flip_shaping_reward = 0.0
        post_flip_potential = None
        near_ground_overspeed_penalty = 0.0

        # 5. Apply continuous post-flip landing-quality shaping.
        if self.flip_completed:
            y_position = float(
                observation[1]
            )

            vertical_speed = float(
                observation[3]
            )

            near_ground_factor = float(
                np.clip(
                    1.0
                    - max(y_position, 0.0)
                    / self.near_ground_height,
                    0.0,
                    1.0,
                )
            )

            downward_speed = max(
                -vertical_speed,
                0.0,
            )

            descent_overspeed = max(
                downward_speed
                - self.safe_touchdown_vertical_speed,
                0.0,
            )

            near_ground_overspeed_penalty = (
                self.near_ground_overspeed_weight
                * near_ground_factor
                * descent_overspeed**2
            )

        # 6. Apply an explicit near-ground descent-overspeed penalty.
        if self.flip_completed:
            post_flip_potential = (
                self._post_flip_potential(observation)
            )

            if self.previous_post_flip_potential is None:
                self.previous_post_flip_potential = (
                    post_flip_potential
                )
            else:
                potential_difference = (
                    self.post_flip_shaping_gamma
                    * post_flip_potential
                    - self.previous_post_flip_potential
                )
                post_flip_shaping_reward = float(
                    np.clip(
                        self.post_flip_shaping_weight
                        * potential_difference,
                        -self.post_flip_shaping_clip,
                        self.post_flip_shaping_clip,
                    )
                )
                self.previous_post_flip_potential = (
                    post_flip_potential
                )

        left_leg_contact = bool(observation[6] > 0.5)
        right_leg_contact = bool(observation[7] > 0.5)

        x_position = float(
            observation[0]
        )

        inside_landing_zone = bool(
            abs(x_position)
            <= self.landing_zone_half_width
        )

        stable_landing = bool(
            terminated
            and float(original_reward) > 0.0
            and left_leg_contact
            and right_leg_contact
            and upright
        )

        # "Safe landing" now means stable and
        # between the landing-zone boundaries.
        landed_safely = bool(
            stable_landing
            and inside_landing_zone
        )

        episode_finished = bool(
            terminated or truncated
        )

        # 7. Classify the terminal outcome and apply its one-off adjustment.
        terminal_adjustment = 0.0
        outcome = "in_progress"

        # Complete success:
        # flipped, settled safely and landed inside the zone.
        if (
            self.flip_completed
            and landed_safely
        ):
            terminal_adjustment = (
                self.flip_landing_bonus
            )

            outcome = (
                "flip_and_safe_landing"
            )

        elif episode_finished:
            # The episode ended before completing the flip.
            if not self.flip_completed:
                if stable_landing:
                    terminal_adjustment = -(
                        self.landing_without_flip_penalty
                    )

                    outcome = (
                        "stable_landing_without_flip"
                    )

                else:
                    terminal_adjustment = -(
                        self.no_flip_terminal_penalty
                    )

                    outcome = (
                        "episode_ended_without_flip"
                    )

            # Step E:
            # The flip was completed and the lander reached
            # the zone, but it crashed instead of settling.
            elif (
                terminated
                and inside_landing_zone
                and not stable_landing
            ):
                terminal_adjustment = -(
                    self.in_zone_crash_penalty
                )

                outcome = (
                    "flip_and_crash_in_zone"
                )

            # A stable landing occurred, but outside the flags.
            elif (
                stable_landing
                and not inside_landing_zone
            ):
                terminal_adjustment = -(
                    self.outside_zone_landing_penalty
                )

                outcome = (
                    "flip_and_stable_landing_"
                    "outside_zone"
                )

            # Other post-flip failures, such as crashing
            # outside the zone or timing out.
            else:
                terminal_adjustment = -(
                    self.failed_landing_penalty
                )

                outcome = (
                    "flip_but_failed_landing"
                )


        original_weight = (
            self.post_flip_original_reward_weight
            if self.flip_completed
            else self.pre_flip_original_reward_weight
        )

        # 8. Combine the original reward, dense shaping and terminal outcome.
        modified_reward = (
            original_weight * float(original_reward)
            + progress_reward
            + completion_reward
            + recovery_reward
            + post_flip_shaping_reward
            - near_ground_overspeed_penalty
            + terminal_adjustment
        )

        info = dict(info)
        info.update(
            {
                "original_reward": float(original_reward),
                "rotation_progress": rotation_progress,
                "rotation_progress_reward": progress_reward,
                "flip_completion_reward": completion_reward,
                "recovery_reward": recovery_reward,
                "post_flip_shaping_reward": (
                    post_flip_shaping_reward
                ),
                "post_flip_potential": post_flip_potential,
                "cumulative_rotation": (
                    self.cumulative_rotation
                ),
                "rotations_completed": (
                    self.cumulative_rotation
                    / (2.0 * math.pi)
                ),
                "flip_completed": self.flip_completed,
                "recovery_completed": (
                    self.recovery_completed
                ),
                "landed_safely": landed_safely,
                "original_reward_weight": original_weight,
                "terminal_adjustment": (
                    terminal_adjustment
                ),
                "flip_landing_bonus": (
                    self.flip_landing_bonus
                    if outcome
                    == "flip_and_safe_landing"
                    else 0.0
                ),
                "custom_outcome": outcome,
                "stable_landing": stable_landing,
                "inside_landing_zone": (
                    inside_landing_zone
                ),
                "horizontal_position": (
                    x_position
                ),
                "near_ground_overspeed_penalty": (
                    near_ground_overspeed_penalty
                ),
                "vertical_speed": float(
                    observation[3]
                ),
            }
        )

        return (
            self._augment_observation(observation),
            modified_reward,
            terminated,
            truncated,
            info,
        )


def make_flip_lander(
    *,
    render_mode: str | None = None,
    reward_config: dict | None = None,
) -> gym.Env:
    """
    Build the exact environment used for training and evaluation.

    Pass the same reward_config to the uploader so the Hub model card
    records the environment accurately.
    """

    base_env = gym.make(
        "LunarLander-v3",
        render_mode=render_mode,
    )

    return FlipLandingRewardWrapper(
        base_env,
        reward_config=reward_config,
    )


### 2.2 Import and reload wrapper

In [ ]:
from copy import deepcopy
from pathlib import Path
import importlib

import flip_landing_reward_wrapper_v2 as flip_wrapper

# Important when the module has previously been imported in this runtime.
importlib.reload(flip_wrapper)

# Default/base reward
DEFAULT_REWARD_CONFIG = (
    flip_wrapper.DEFAULT_REWARD_CONFIG
)
base_reward_config = deepcopy(
    DEFAULT_REWARD_CONFIG
)
base_reward_config[
    "post_flip_shaping_gamma"
] = 0.999

# Reward wrapper
FlipLandingRewardWrapper = (
    flip_wrapper.FlipLandingRewardWrapper
)
make_flip_lander = (
    flip_wrapper.make_flip_lander
)
REWARD_WRAPPER_PATH = Path(
    "/content/flip_landing_reward_wrapper_v2.py"
)

### 2.3 Curriculum reward configurations

Each phase updates the same PPO policy, but changes the reward emphasis:

- **Phase A:** discover and complete the flip.
- **Phase B:** recover upright and attempt a landing.
- **Phase C:** reach the landing zone.
- **Phase D:** reduce descent speed and land softly.

In [ ]:
# Phase A inherits the base defaults and emphasises discovering the flip.
flip_acquisition_config = {
    **base_reward_config,

    # Make discovering the flip worthwhile.
    "rotation_progress_bonus": 300.0,
    "flip_completion_bonus": 500.0,

    # Retain some position and velocity guidance,
    # but prevent ordinary landing from winning.
    "pre_flip_original_reward_weight": 0.15,
    "post_flip_original_reward_weight": 1.0,

    "landing_without_flip_penalty": 300.0,
    "no_flip_terminal_penalty": 300.0,

    # During acquisition, a flip followed by a crash
    # must still be recognised as progress.
    "failed_landing_penalty": 0.0,

    "recovery_bonus": 100.0,
    "post_flip_shaping_weight": 0.5,

    "flip_landing_bonus": 1_000.0,

    # Disable objectives that belong to later curriculum stages.
    "post_flip_vertical_speed_weight": 0.0,
    "near_ground_overspeed_weight": 0.0,
    "outside_zone_landing_penalty": 0.0,
    "in_zone_crash_penalty": 0.0,
}

# Phase B inherits Phase A and adds upright recovery and landing pressure.
flip_landing_config = {
    **flip_acquisition_config,

    # Keep the flip rewards. Reducing them risks
    # catastrophic forgetting of the rotation.
    "rotation_progress_bonus": 300.0,
    "flip_completion_bonus": 500.0,

    # The requested post-flip landing multiplier.
    "post_flip_original_reward_weight": 3.0,

    "recovery_bonus": 250.0,
    "post_flip_shaping_weight": 1.5,

    # Once the flip is learned, crashes should
    # become significantly unattractive.
    "failed_landing_penalty": 400.0,

    # Exclusive reward for completing both stages.
    "flip_landing_bonus": 1_500.0,

    # Phase B treats all post-flip crashes similarly.
    "post_flip_vertical_speed_weight": 30.0,
    "near_ground_overspeed_weight": 0.0,
    "outside_zone_landing_penalty": 400.0,
    "in_zone_crash_penalty": 400.0,
}

# Phase C inherits Phase B and makes landing-zone acquisition explicit.
flip_in_zone_config = {
    **flip_landing_config,

    # The current flip and recovery rewards remain
    # unchanged to preserve the learned rotation.
    "rotation_progress_bonus": 300.0,
    "flip_completion_bonus": 500.0,
    "recovery_bonus": 250.0,

    # Landing zone is approximately between
    # observation x=-0.20 and x=+0.20.
    "landing_zone_half_width": 0.20,

    # Stable off-zone landings must no longer
    # receive the success return.
    "outside_zone_landing_penalty": 1_000.0,

    # Stronger post-flip pad acquisition.
    "post_flip_center_weight": 120.0,
    "post_flip_zone_excess_weight": 200.0,

    # Allow purposeful lateral movement towards
    # the centre rather than demanding vx=0.
    "post_flip_horizontal_speed_weight": 40.0,
    "post_flip_target_vx_gain": 0.50,
    "post_flip_max_target_vx": 0.35,

    # Slightly stronger recovery shaping.
    "post_flip_shaping_weight": 2.0,
    "post_flip_shaping_clip": 20.0,

    # Keep the exclusive in-zone success bonus.
    "flip_landing_bonus": 1_500.0,

    # Phase C rewards pad acquisition but does not yet apply
    # the dedicated soft-touchdown overspeed penalty.
    "post_flip_vertical_speed_weight": 30.0,
    "near_ground_overspeed_weight": 0.0,
    "outside_zone_landing_penalty": 1_000.0,
    "in_zone_crash_penalty": 400.0,
}

# Phase D inherits Phase C and refines the final descent and touchdown
flip_soft_landing_config = {
    **flip_in_zone_config,

    # Preserve the learned rotation and recovery behaviour.
    "rotation_progress_bonus": 300.0,
    "flip_completion_bonus": 500.0,
    "recovery_bonus": 250.0,

    # A stable off-zone landing is undesirable, but is still preferable
    # to crashing after entering the landing zone.
    "outside_zone_landing_penalty": 500.0,
    "failed_landing_penalty": 800.0,
    "in_zone_crash_penalty": 1_200.0,

    # Reduce the Phase C emphasis on aggressive pad acquisition.
    "post_flip_center_weight": 80.0,
    "post_flip_zone_excess_weight": 120.0,

    # Prioritise controlled horizontal and vertical motion.
    "post_flip_horizontal_speed_weight": 50.0,
    "post_flip_vertical_speed_weight": 100.0,
    "post_flip_angle_weight": 70.0,
    "post_flip_angular_speed_weight": 40.0,
    "post_flip_leg_contact_weight": 30.0,

    # Descend at a controlled rate while high, then slow near touchdown.
    "post_flip_target_vy_high": -0.45,
    "post_flip_target_vy_near_ground": -0.12,
    "near_ground_height": 0.60,
    "safe_touchdown_vertical_speed": 0.18,
    "near_ground_overspeed_weight": 120.0,

    # Avoid unnecessary last-second lateral corrections.
    "post_flip_horizontal_deadband": 0.08,

    # Keep dense shaping large enough to provide immediate feedback.
    "post_flip_shaping_weight": 2.0,
    "post_flip_shaping_clip": 30.0,

    # Exclusive reward for completing the complete task.
    "flip_landing_bonus": 2_000.0,
}

### 2.4 Environment factories

In [ ]:
def make_flip_acquisition_lander(
    *,
    render_mode: str | None = None,
):
    return make_flip_lander(
        render_mode=render_mode,
        reward_config=(
            flip_acquisition_config
        ),
    )


def make_flip_landing_lander(
    *,
    render_mode: str | None = None,
):
    return make_flip_lander(
        render_mode=render_mode,
        reward_config=(
            flip_landing_config
        ),
    )

def make_flip_in_zone_lander(
    *,
    render_mode: str | None = None,
):
    return make_flip_lander(
        render_mode=render_mode,
        reward_config=(
            flip_in_zone_config
        ),
    )

def make_flip_soft_landing_lander(
    *,
    render_mode: str | None = None,
):
    """Create the Phase D soft-touchdown environment."""

    return make_flip_lander(
        render_mode=render_mode,
        reward_config=(
            flip_soft_landing_config
        ),
    )

### 2.5 Environment smoke tests

This test constructs all four curriculum environments and confirms that
they expose the same 11-value observation space and four-action discrete
action space. That compatibility allows one PPO checkpoint to be continued
across all curriculum phases.

In [ ]:
# Verify that every phase factory creates an environment compatible
# with the same saved 11-observation PPO policy.

phase_factories = {
    "Phase A": make_flip_acquisition_lander,
    "Phase B": make_flip_landing_lander,
    "Phase C": make_flip_in_zone_lander,
    "Phase D": make_flip_soft_landing_lander,
}

for phase_name, env_factory in phase_factories.items():
    test_env = env_factory()

    try:
        observation, info = test_env.reset(
            seed=42
        )

        assert observation.shape == (11,)
        assert test_env.observation_space.shape == (11,)
        assert test_env.action_space.n == 4

        print(
            f"{phase_name} environment passed:",
            observation.shape,
            test_env.action_space,
        )

    finally:
        test_env.close()

print("All environment factories passed.")

## 3. Shared training infrastructure

### 3.1 Persistent Hub checkpoint callback

In [ ]:
from pathlib import Path

from huggingface_hub import HfApi
from stable_baselines3.common.callbacks import (
    BaseCallback,
)

class HubCheckpointCallback(BaseCallback):
    """
    Persist phase-labelled checkpoints to Hugging Face.

    `phase_timesteps` counts only training performed in this phase.
    `total_timesteps` retains SB3's cumulative model timestep count.
    """

    def __init__(
        self,
        *,
        repo_id: str,
        phase_name: str,
        every_timesteps: int = 500_000,
        local_root: str | Path = (
            "/content/hf-checkpoints"
        ),
        verbose: int = 1,
    ):
        super().__init__(verbose)

        if every_timesteps <= 0:
            raise ValueError(
                "every_timesteps must be positive."
            )

        self.repo_id = repo_id
        self.phase_name = (
            phase_name
            .strip()
            .lower()
            .replace(" ", "_")
        )
        self.every_timesteps = int(
            every_timesteps
        )
        self.local_directory = (
            Path(local_root)
            / self.phase_name
        )
        self.api = HfApi()

        self.phase_start_timestep = 0
        self.next_upload = self.every_timesteps

    def _on_training_start(self) -> None:
        self.local_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        self.phase_start_timestep = int(
            self.model.num_timesteps
        )

        self.next_upload = (
            self.every_timesteps
        )

    def _on_step(self) -> bool:
        total_timesteps = int(
            self.num_timesteps
        )

        phase_timesteps = (
            total_timesteps
            - self.phase_start_timestep
        )

        if phase_timesteps < self.next_upload:
            return True

        filename = (
            f"{self.phase_name}_"
            f"{phase_timesteps:09d}_phase_steps_"
            f"{total_timesteps:09d}_total_steps.zip"
        )

        local_path = (
            self.local_directory
            / filename
        )

        self.model.save(
            str(local_path)
        )

        try:
            self.api.upload_file(
                path_or_fileobj=str(
                    local_path
                ),
                path_in_repo=(
                    "training-checkpoints/"
                    f"{self.phase_name}/"
                    f"{filename}"
                ),
                repo_id=self.repo_id,
                repo_type="model",
                commit_message=(
                    f"Save {self.phase_name} "
                    f"checkpoint at "
                    f"{phase_timesteps:,} phase steps"
                ),
            )

            if self.verbose:
                print(
                    "Uploaded checkpoint:",
                    filename,
                )

        except Exception as exc:
            # A temporary Hub failure should not destroy
            # a multi-hour training run.
            print(
                "Checkpoint upload failed:",
                repr(exc),
            )

        finally:
            while (
                self.next_upload
                <= phase_timesteps
            ):
                self.next_upload += (
                    self.every_timesteps
                )

        return True

### 3.2 Model loading

In [ ]:
def load_ppo_model(
    model_or_path: PPO | str | Path,
    *,
    device: str = "cpu",
) -> PPO:
    """Return an existing PPO object or load one from disk."""

    if isinstance(model_or_path, PPO):
        return model_or_path

    model_path = Path(model_or_path)

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model not found: {model_path}"
        )

    return PPO.load(
        str(model_path),
        device=device,
    )


### 3.3 Training function

In [ ]:
from collections.abc import Callable, Sequence
from pathlib import Path
from typing import Any
import json

import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    EvalCallback,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor


def train_ppo_experiment(
    *,
    experiment_name: str,
    total_timesteps: int,
    env_id: str = "LunarLander-v3",
    env_factory: Callable[[], gym.Env] | None = None,
    env_kwargs: dict[str, Any] | None = None,
    n_envs: int = 16,
    seed: int = 42,
    actor_layers: Sequence[int] = (64, 64),
    critic_layers: Sequence[int] = (64, 64),
    learning_rate: float = 3e-4,
    n_steps: int = 1024,
    batch_size: int = 64,
    n_epochs: int = 4,
    gamma: float = 0.999,
    gae_lambda: float = 0.98,
    ent_coef: float = 0.01,
    clip_range: float = 0.2,
    eval_every_timesteps: int = 50_000,
    checkpoint_every_timesteps: int = 100_000,
    n_eval_episodes: int = 20,
    output_root: str | Path = "/content/rl-experiments",
    device: str = "cpu",
    verbose: int = 1,
    extra_callbacks: Sequence[
        BaseCallback
    ] | None = None,
    initial_model_path: (
        str | Path | None
    ) = None,
) -> dict[str, Any]:
    """
    Train a PPO actor–critic model.

    env_factory
        Optional callable that returns a custom Gymnasium environment.
        Use this for the flip-reward environment.

    The function saves:
    - periodic safety checkpoints;
    - the checkpoint with the highest evaluation mean;
    - the model from the final training timestep;
    - the complete training configuration.
    """

    if not experiment_name.strip():
        raise ValueError(
            "experiment_name must not be empty."
        )

    if total_timesteps <= 0:
        raise ValueError(
            "total_timesteps must be positive."
        )

    if n_envs <= 0:
        raise ValueError(
            "n_envs must be positive."
        )

    env_kwargs = dict(env_kwargs or {})

    if env_factory is not None and env_kwargs:
        raise ValueError(
            "Use either env_factory or env_kwargs, not both. "
            "Put custom environment arguments inside the factory."
        )

    output_dir = (
        Path(output_root)
        / experiment_name
    )
    best_model_dir = (
        output_dir
        / "best_model"
    )
    checkpoint_dir = (
        output_dir
        / "checkpoints"
    )
    evaluation_dir = (
        output_dir
        / "evaluations"
    )

    for directory in (
        best_model_dir,
        checkpoint_dir,
        evaluation_dir,
    ):
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    configuration = {
        "experiment_name": experiment_name,
        "env_id": env_id,
        "environment_factory": (
            getattr(
                env_factory,
                "__name__",
                None,
            )
            if env_factory is not None
            else None
        ),
        "env_kwargs": env_kwargs,
        "total_timesteps": total_timesteps,
        "n_envs": n_envs,
        "seed": seed,
        "actor_layers": list(actor_layers),
        "critic_layers": list(critic_layers),
        "learning_rate": learning_rate,
        "n_steps": n_steps,
        "batch_size": batch_size,
        "n_epochs": n_epochs,
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "ent_coef": ent_coef,
        "clip_range": clip_range,
        "eval_every_timesteps": (
            eval_every_timesteps
        ),
        "checkpoint_every_timesteps": (
            checkpoint_every_timesteps
        ),
        "n_eval_episodes": n_eval_episodes,
        "device": device,
        "initial_model_path": (
            str(initial_model_path)
            if initial_model_path is not None
            else None
        ),
    }

    config_path = (
        output_dir
        / "training_config.json"
    )

    config_path.write_text(
        json.dumps(
            configuration,
            indent=2,
        ),
        encoding="utf-8",
    )

    # Create standard or customised environments.
    if env_factory is None:
        train_env = make_vec_env(
            env_id,
            n_envs=n_envs,
            seed=seed,
            env_kwargs=env_kwargs,
        )

        eval_env = Monitor(
            gym.make(
                env_id,
                **env_kwargs,
            )
        )

    else:
        train_env = make_vec_env(
            env_factory,
            n_envs=n_envs,
            seed=seed,
        )

        eval_env = Monitor(
            env_factory()
        )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(
            best_model_dir
        ),
        log_path=str(
            evaluation_dir
        ),
        eval_freq=max(
            eval_every_timesteps // n_envs,
            1,
        ),
        n_eval_episodes=n_eval_episodes,
        deterministic=True,
        render=False,
        verbose=1,
    )

    checkpoint_callback = CheckpointCallback(
        save_freq=max(
            checkpoint_every_timesteps // n_envs,
            1,
        ),
        save_path=str(
            checkpoint_dir
        ),
        name_prefix=experiment_name,
        verbose=2,
    )

    callbacks = [
        eval_callback,
        checkpoint_callback,
        *(extra_callbacks or []),
    ]

    policy_kwargs = {
        "net_arch": {
            "pi": list(actor_layers),
            "vf": list(critic_layers),
        }
    }

    if initial_model_path is None:
        model = PPO(
            policy="MlpPolicy",
            env=train_env,
            learning_rate=learning_rate,
            n_steps=n_steps,
            batch_size=batch_size,
            n_epochs=n_epochs,
            gamma=gamma,
            gae_lambda=gae_lambda,
            ent_coef=ent_coef,
            clip_range=clip_range,
            policy_kwargs=policy_kwargs,
            device=device,
            seed=seed,
            verbose=verbose,
        )

        reset_num_timesteps = True

    else:
        model = PPO.load(
            str(initial_model_path),
            env=train_env,
            device=device,

            # Override selected training settings
            # for the refinement phase.
            learning_rate=learning_rate,
            n_steps=n_steps,
            batch_size=batch_size,
            n_epochs=n_epochs,
            gamma=gamma,
            gae_lambda=gae_lambda,
            ent_coef=ent_coef,
            clip_range=clip_range,
        )

      reset_num_timesteps = False

    try:
        model.learn(
            total_timesteps=total_timesteps,
            callback=callbacks,
            progress_bar=True,
            reset_num_timesteps=(
                reset_num_timesteps
            ),
        )

        final_model_stem = (
            output_dir
            / "final_model"
        )

        model.save(
            str(final_model_stem)
        )

    finally:
        train_env.close()
        eval_env.close()

    best_model_path = (
        best_model_dir
        / "best_model.zip"
    )
    final_model_path = (
        output_dir
        / "final_model.zip"
    )
    evaluations_path = (
        evaluation_dir
        / "evaluations.npz"
    )

    if not best_model_path.exists():
        raise FileNotFoundError(
            "EvalCallback did not create best_model.zip. "
            "Check that training reached at least one "
            "evaluation interval."
        )

    print("Best model:", best_model_path)
    print("Final model:", final_model_path)
    print("Configuration:", config_path)

    return {
        "experiment_name": experiment_name,
        "output_dir": output_dir,
        "best_model_path": best_model_path,
        "final_model_path": final_model_path,
        "evaluations_path": evaluations_path,
        "config_path": config_path,
        "configuration": configuration,
    }

### 3.4 Model evaluation

In [ ]:
import numpy as np
import pandas as pd

def evaluate_flip_model(
    model_or_path,
    *,
    reward_config: dict,
    n_episodes: int = 100,
    starting_seed: int = 20_000,
    deterministic: bool = True,
):
    """
    Evaluate the flip-recover-land agent over fixed seeds.
    """

    if n_episodes <= 0:
        raise ValueError(
            "n_episodes must be positive."
        )

    model = load_ppo_model(
        model_or_path
    )

    episode_rows = []

    for episode_number in range(
        n_episodes
    ):
        seed = (
            starting_seed
            + episode_number
        )

        env = make_flip_lander(
            reward_config=reward_config
        )

        try:
            observation, _ = env.reset(
                seed=seed
            )

            terminated = False
            truncated = False

            shaped_reward_total = 0.0
            original_reward_total = 0.0
            episode_steps = 0
            final_info = {}

            while not (
                terminated or truncated
            ):
                action, _ = model.predict(
                    observation,
                    deterministic=deterministic,
                )

                # LunarLander-v3 uses Discrete(4).
                action = int(
                    np.asarray(
                        action
                    ).item()
                )

                (
                    observation,
                    shaped_reward,
                    terminated,
                    truncated,
                    info,
                ) = env.step(action)

                shaped_reward_total += float(
                    shaped_reward
                )

                original_reward_total += float(
                    info.get(
                        "original_reward",
                        shaped_reward,
                    )
                )

                episode_steps += 1
                final_info = dict(info)

            flip_completed = bool(
                final_info.get(
                    "flip_completed",
                    False,
                )
            )

            recovery_completed = bool(
                final_info.get(
                    "recovery_completed",
                    False,
                )
            )

            landed_safely = bool(
                final_info.get(
                    "landed_safely",
                    False,
                )
            )

            custom_outcome = str(
                final_info.get(
                    "custom_outcome",
                    "unknown",
                )
            )

            flip_and_landed = (
                custom_outcome
                == "flip_and_safe_landing"
            )

            stable_landing = bool(
                final_info.get(
                    "stable_landing",
                    False,
                )
            )

            inside_landing_zone = bool(
                final_info.get(
                    "inside_landing_zone",
                    False,
                )
            )

            episode_rows.append(
                {
                    "seed": seed,
                    "shaped_reward": (
                        shaped_reward_total
                    ),
                    "original_reward": (
                        original_reward_total
                    ),
                    "steps": episode_steps,
                    "flip_completed": (
                        flip_completed
                    ),
                    "recovery_completed": (
                        recovery_completed
                    ),
                    "landed_safely": (
                        landed_safely
                    ),
                    "flip_and_landed": (
                        flip_and_landed
                    ),
                    "flip_bonus": float(
                        final_info.get(
                            "flip_landing_bonus",
                            0.0,
                        )
                    ),
                    "rotations_completed": float(
                        final_info.get(
                            "rotations_completed",
                            0.0,
                        )
                    ),
                    "terminal_adjustment": float(
                        final_info.get(
                            "terminal_adjustment",
                            0.0,
                        )
                    ),
                    "custom_outcome": (
                        custom_outcome
                    ),
                    "stable_landing": stable_landing,
                    "inside_landing_zone": (
                        inside_landing_zone
                    ),
                    "outside_zone_landing": (
                        stable_landing
                        and not inside_landing_zone
                    ),
                    "in_zone_crash": (
                        custom_outcome
                        == "flip_and_crash_in_zone"
                    ),
                }
            )

        finally:
            env.close()

    episodes = pd.DataFrame(
        episode_rows
    )

    shaped_rewards = episodes[
        "shaped_reward"
    ].to_numpy(dtype=float)

    original_rewards = episodes[
        "original_reward"
    ].to_numpy(dtype=float)

    flip_mask = episodes[
        "flip_completed"
    ]

    if flip_mask.any():
        recovery_given_flip_rate = float(
            episodes.loc[
                flip_mask,
                "recovery_completed",
            ].mean()
        )

        landing_given_flip_rate = float(
            episodes.loc[
                flip_mask,
                "flip_and_landed",
            ].mean()
        )
    else:
        recovery_given_flip_rate = 0.0
        landing_given_flip_rate = 0.0

    summary = {
        "n_episodes": int(n_episodes),

        "mean_shaped_reward": float(
            shaped_rewards.mean()
        ),
        "std_shaped_reward": float(
            shaped_rewards.std()
        ),
        "shaped_course_score": float(
            shaped_rewards.mean()
            - shaped_rewards.std()
        ),
        "min_shaped_reward": float(
            shaped_rewards.min()
        ),
        "max_shaped_reward": float(
            shaped_rewards.max()
        ),

        "mean_original_reward": float(
            original_rewards.mean()
        ),
        "std_original_reward": float(
            original_rewards.std()
        ),
        "original_course_score": float(
            original_rewards.mean()
            - original_rewards.std()
        ),
        "min_original_reward": float(
            original_rewards.min()
        ),
        "max_original_reward": float(
            original_rewards.max()
        ),

        "original_reward_200_rate": float(
            np.mean(
                original_rewards >= 200.0
            )
        ),
        "flip_completion_rate": float(
            episodes[
                "flip_completed"
            ].mean()
        ),
        "recovery_completion_rate": float(
            episodes[
                "recovery_completed"
            ].mean()
        ),
        "recovery_given_flip_rate": (
            recovery_given_flip_rate
        ),
        "safe_landing_rate": float(
            episodes[
                "landed_safely"
            ].mean()
        ),
        "flip_landing_rate": float(
            episodes[
                "flip_and_landed"
            ].mean()
        ),
        "landing_given_flip_rate": (
            landing_given_flip_rate
        ),
        "stable_landing_rate": float(
            episodes[
                "stable_landing"
            ].mean()
        ),

        "inside_landing_zone_rate": float(
            episodes[
                "inside_landing_zone"
            ].mean()
        ),

        "outside_zone_landing_rate": float(
            episodes[
                "outside_zone_landing"
            ].mean()
        ),

        "in_zone_crash_rate": float(
            episodes[
                "in_zone_crash"
            ].mean()
        ),
    }

    print("Flip model evaluation")
    print("---------------------")
    print(
        "Mean shaped reward:",
        f"{summary['mean_shaped_reward']:.2f}",
    )
    print(
        "Mean original reward:",
        f"{summary['mean_original_reward']:.2f}",
    )
    print(
        "Completed a full rotation:",
        f"{summary['flip_completion_rate']:.1%}",
    )
    print(
        "Recovered after the flip:",
        f"{summary['recovery_completion_rate']:.1%}",
    )
    print(
        "Recovery rate given a flip:",
        f"{summary['recovery_given_flip_rate']:.1%}",
    )
    print(
        "Landed safely:",
        f"{summary['safe_landing_rate']:.1%}",
    )
    print(
        "Flipped and landed safely:",
        f"{summary['flip_landing_rate']:.1%}",
    )
    print(
        "Landing rate given a flip:",
        f"{summary['landing_given_flip_rate']:.1%}",
    )

    return {
        "summary": summary,
        "episodes": episodes,
    }

### 3.5 Checkpoint filename parsing

In [ ]:
import re
from pathlib import Path


def checkpoint_step(path: Path) -> int:
    """Extract the saved timestep from an SB3 checkpoint filename."""

    match = re.search(
        r"_(\d+)_steps$",
        path.stem,
    )

    return (
        int(match.group(1))
        if match
        else -1
    )

### 3.6 Candidate checkpoint evaluation

In [ ]:
def evaluate_phase_candidates(
    run: dict,
    *,
    reward_config: dict,
    n_episodes: int = 30,
    starting_seed: int = 20_000,
) -> pd.DataFrame:
    """
    Evaluate best, final and periodic checkpoints on identical seeds.

    EvalCallback's best_model.zip is selected by mean shaped reward.
    This function lets us select using task-specific metrics instead.
    """

    checkpoint_paths = sorted(
        (
            run["output_dir"]
            / "checkpoints"
        ).glob("*.zip"),
        key=checkpoint_step,
    )

    candidate_paths = [
        Path(run["best_model_path"]),
        Path(run["final_model_path"]),
        *checkpoint_paths,
    ]

    # Preserve order while removing duplicates and missing paths.
    candidate_paths = list(
        dict.fromkeys(
            path
            for path in candidate_paths
            if path.exists()
        )
    )

    if not candidate_paths:
        raise RuntimeError(
            "No model candidates were found."
        )

    rows = []

    for candidate_path in candidate_paths:
        print(
            "Evaluating:",
            candidate_path.name,
        )

        evaluation = evaluate_flip_model(
            candidate_path,
            reward_config=reward_config,
            n_episodes=n_episodes,
            starting_seed=starting_seed,
        )

        rows.append(
            {
                "model_path": str(
                    candidate_path
                ),
                **evaluation["summary"],
            }
        )

    return pd.DataFrame(rows)

### 3.7 Selected-phase artifact upload

In [ ]:
def upload_selected_phase_artifacts(
    *,
    phase_name: str,
    model_path: str | Path,
    run: dict,
    reward_config: dict,
    candidate_results: pd.DataFrame,
    repo_id: str,
) -> None:
    """Upload the selected model and its reproducibility metadata."""

    phase_name = (
        phase_name
        .strip()
        .lower()
        .replace(" ", "_")
    )

    artifact_directory = (
        Path("/content/selected-phase-artifacts")
        / phase_name
    )

    artifact_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    reward_config_path = (
        artifact_directory
        / "reward_config.json"
    )

    candidate_results_path = (
        artifact_directory
        / "candidate_results.csv"
    )

    reward_config_path.write_text(
        json.dumps(
            reward_config,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    candidate_results.to_csv(
        candidate_results_path,
        index=False,
    )

    api = HfApi()

    api.upload_file(
        path_or_fileobj=str(
            model_path
        ),
        path_in_repo=(
            "selected-models/"
            f"{phase_name}_best.zip"
        ),
        repo_id=repo_id,
        repo_type="model",
        commit_message=(
            f"Save selected {phase_name} model"
        ),
    )

    api.upload_file(
        path_or_fileobj=str(
            run["config_path"]
        ),
        path_in_repo=(
            "selected-models/"
            f"{phase_name}_training_config.json"
        ),
        repo_id=repo_id,
        repo_type="model",
        commit_message=(
            f"Save {phase_name} training config"
        ),
    )

    api.upload_file(
        path_or_fileobj=str(
            reward_config_path
        ),
        path_in_repo=(
            "selected-models/"
            f"{phase_name}_reward_config.json"
        ),
        repo_id=repo_id,
        repo_type="model",
        commit_message=(
            f"Save {phase_name} reward config"
        ),
    )

    api.upload_file(
        path_or_fileobj=str(
            candidate_results_path
        ),
        path_in_repo=(
            "selected-models/"
            f"{phase_name}_candidate_results.csv"
        ),
        repo_id=repo_id,
        repo_type="model",
        commit_message=(
            f"Save {phase_name} candidate results"
        ),
    )

### 3.8 Replay recording

In [ ]:
from pathlib import Path
import shutil

import gymnasium as gym


def record_flip_replay(
    model_or_path,
    *,
    output_path: str | Path,
    seed: int,
    reward_config: dict,
    deterministic: bool = True,
):
    """
    Record one complete episode from the custom flip environment.
    """

    model = load_ppo_model(
        model_or_path
    )

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_directory = (
        output_path.parent
        / "_flip_video_recording"
    )

    if temporary_directory.exists():
        shutil.rmtree(
            temporary_directory
        )

    temporary_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    video_env = make_flip_lander(
        render_mode="rgb_array",
        reward_config=reward_config,
    )

    video_env = gym.wrappers.RecordVideo(
        video_env,
        video_folder=str(
            temporary_directory
        ),
        episode_trigger=(
            lambda episode_number:
            episode_number == 0
        ),
        video_length=0,
        name_prefix=(
            "ppo-lunarlander-flip"
        ),
        disable_logger=True,
    )

    shaped_reward_total = 0.0
    original_reward_total = 0.0
    episode_steps = 0

    final_info = {}

    try:
        observation, info = video_env.reset(
            seed=int(seed)
        )

        terminated = False
        truncated = False

        while not (
            terminated or truncated
        ):
            action, _ = model.predict(
                observation,
                deterministic=deterministic,
            )

            action = int(
                np.asarray(
                    action
                ).item()
            )

            (
                observation,
                shaped_reward,
                terminated,
                truncated,
                info,
            ) = video_env.step(action)

            shaped_reward_total += float(
                shaped_reward
            )

            original_reward_total += float(
                info.get(
                    "original_reward",
                    shaped_reward,
                )
            )

            episode_steps += 1
            final_info = dict(info)

    finally:
        video_env.close()

    generated_videos = list(
        temporary_directory.glob(
            "*.mp4"
        )
    )

    if not generated_videos:
        raise FileNotFoundError(
            "Gymnasium did not generate "
            "an MP4 replay."
        )

    generated_video = max(
        generated_videos,
        key=lambda path:
        path.stat().st_mtime,
    )

    shutil.copy2(
        generated_video,
        output_path,
    )

    shutil.rmtree(
        temporary_directory,
        ignore_errors=True,
    )

    replay_details = {
        "path": str(output_path),
        "seed": int(seed),
        "shaped_reward": float(
            shaped_reward_total
        ),
        "original_reward": float(
            original_reward_total
        ),
        "steps": int(
            episode_steps
        ),
        "flip_completed": bool(
            final_info.get(
                "flip_completed",
                False,
            )
        ),
        "landed_safely": bool(
            final_info.get(
                "landed_safely",
                False,
            )
        ),
        "rotations_completed": float(
            final_info.get(
                "rotations_completed",
                0.0,
            )
        ),
        "recovery_completed": bool(
            final_info.get(
                "recovery_completed",
                False,
            )
        ),
        "custom_outcome": str(
            final_info.get(
                "custom_outcome",
                "unknown",
            )
        ),
        "flip_and_landed": (
            final_info.get(
                "custom_outcome"
            )
            == "flip_and_safe_landing"
        ),
    }

    print("Replay saved:", output_path)
    print(
        "Shaped reward:",
        f"{shaped_reward_total:.2f}",
    )
    print(
        "Original reward:",
        f"{original_reward_total:.2f}",
    )
    print(
        "Completed a flip:",
        replay_details[
            "flip_completed"
        ],
    )
    print(
        "Landed safely:",
        replay_details[
            "landed_safely"
        ],
    )
    print(
        "Flipped and landed:",
        replay_details[
            "flip_and_landed"
        ],
    )
    print(
        "Rotations completed:",
        f"{replay_details['rotations_completed']:.2f}",
    )

    return replay_details

### 3.9 Final Hub publication and model-card generation

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import textwrap

from huggingface_hub import (
    HfApi,
    ModelCard,
    ModelCardData,
)


DEFAULT_CHANGE_NOTES = [
    (
        "Published the selected curriculum checkpoint with its "
        "training configuration, reward configuration, evaluation "
        "results and replay."
    ),
]


def _markdown_value(value) -> str:
    """Format a Python value for a Markdown table."""

    if isinstance(value, bool):
        return "true" if value else "false"

    if isinstance(value, float):
        return f"{value:g}"

    return str(value)


def _markdown_table(mapping: dict) -> str:
    """Convert a dictionary into a two-column Markdown table."""

    rows = [
        "| Parameter | Value |",
        "|---|---:|",
    ]

    for key, value in mapping.items():
        rows.append(
            f"| `{key}` | {_markdown_value(value)} |"
        )

    return "\n".join(rows)


def upload_flip_model_to_hub(
    model_or_path,
    *,
    repo_id: str,
    evaluation: dict,
    replay: dict,
    training_config_path: str | Path,
    reward_config: dict,
    reward_wrapper_path: str | Path,
    model_filename: str = (
        "ppo-LunarLander-v3-flip-128x128.zip"
    ),
    reward_version: str = "v4-soft-touchdown",
    change_notes: list[str] | None = None,
    commit_message: str = (
        "Publish PPO flip-and-land model"
    ),
    private: bool = False,
):
    """
    Publish the selected PPO model and its reproducibility artifacts.

    The upload contains:
    - the selected PPO checkpoint;
    - training and reward configurations;
    - the custom reward wrapper;
    - episode-level evaluation results;
    - JSON evaluation metadata;
    - MP4 and GIF replays;
    - a validated Hugging Face model card.
    """

    import subprocess

    if not reward_config:
        raise ValueError(
            "reward_config must contain the exact "
            "configuration used for training."
        )

    if (
        "summary" not in evaluation
        or "episodes" not in evaluation
    ):
        raise ValueError(
            "evaluation must contain 'summary' "
            "and 'episodes'."
        )

    if "path" not in replay:
        raise ValueError(
            "replay must contain its local file path."
        )

    model = load_ppo_model(
        model_or_path
    )

    summary = dict(
        evaluation["summary"]
    )

    episode_results = (
        evaluation["episodes"].copy()
    )

    replay_details = dict(replay)

    replay_path = Path(
        replay_details["path"]
    )

    training_config_path = Path(
        training_config_path
    )

    reward_wrapper_path = Path(
        reward_wrapper_path
    )

    required_paths = {
        "Replay": replay_path,
        "Training configuration": (
            training_config_path
        ),
        "Reward wrapper": (
            reward_wrapper_path
        ),
    }

    for label, path in required_paths.items():
        if not path.exists():
            raise FileNotFoundError(
                f"{label} not found: {path}"
            )

    training_config = json.loads(
        training_config_path.read_text(
            encoding="utf-8"
        )
    )

    # Build the complete repository locally before
    # making any changes to the Hub repository.
    staging_directory = (
        Path("/content/hf-flip-model")
        / repo_id.replace("/", "__")
    )

    if staging_directory.exists():
        shutil.rmtree(
            staging_directory
        )

    staging_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ------------------------------------------------------------------
    # Model checkpoint
    # ------------------------------------------------------------------

    saved_model_path = (
        staging_directory
        / model_filename
    )

    model.save(
        str(saved_model_path)
    )

    if not saved_model_path.exists():
        raise FileNotFoundError(
            "The PPO model was not saved at "
            f"{saved_model_path}"
        )

    # ------------------------------------------------------------------
    # Replay files
    # ------------------------------------------------------------------

    replay_mp4_path = (
        staging_directory
        / "replay.mp4"
    )

    replay_gif_path = (
        staging_directory
        / "replay.gif"
    )

    shutil.copy2(
        replay_path,
        replay_mp4_path,
    )

    # Hugging Face model cards reliably display the GIF.
    # Clicking the GIF opens the higher-quality MP4.
    gif_filter = (
        "fps=12,"
        "scale=640:-1:flags=lanczos,"
        "split[s0][s1];"
        "[s0]palettegen=max_colors=128[p];"
        "[s1][p]paletteuse"
    )

    gif_result = subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i",
            str(replay_mp4_path),
            "-filter_complex",
            gif_filter,
            "-loop",
            "0",
            str(replay_gif_path),
        ],
        check=False,
        capture_output=True,
        text=True,
    )

    if gif_result.returncode != 0:
        raise RuntimeError(
            "Failed to create replay.gif:\n"
            + gif_result.stderr
        )

    if not replay_gif_path.exists():
        raise FileNotFoundError(
            "ffmpeg did not create replay.gif."
        )

    # ------------------------------------------------------------------
    # Reproducibility artifacts
    # ------------------------------------------------------------------

    shutil.copy2(
        training_config_path,
        staging_directory
        / "training_config.json",
    )

    shutil.copy2(
        reward_wrapper_path,
        staging_directory
        / "flip_landing_reward_wrapper.py",
    )

    (
        staging_directory
        / "reward_config.json"
    ).write_text(
        json.dumps(
            reward_config,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    episode_results.to_csv(
        staging_directory
        / "episode_results.csv",
        index=False,
    )

    notes = list(
        change_notes
        if change_notes is not None
        else DEFAULT_CHANGE_NOTES
    )

    results_payload = {
        "environment": {
            "base_environment": (
                "LunarLander-v3"
            ),
            "custom_objective": (
                "Complete a full directed rotation, "
                "recover upright, enter the landing "
                "zone and land safely."
            ),
            "reward_version": reward_version,
            "observation_space": {
                "base_dimensions": 8,
                "added_dimensions": [
                    "signed_rotation_progress",
                    "flip_completed",
                    "recovery_completed",
                ],
                "wrapped_dimensions": 11,
            },
            "reward_config": reward_config,
            "change_notes": notes,
        },
        "evaluation": summary,
        "evaluation_seeds": (
            episode_results["seed"]
            .astype(int)
            .tolist()
        ),
        "replay": {
            key: value
            for key, value
            in replay_details.items()
            if key != "path"
        },
    }

    (
        staging_directory
        / "results.json"
    ).write_text(
        json.dumps(
            results_payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    config_payload = {
        "algorithm": "PPO",
        "library": "stable-baselines3",
        "base_environment": (
            "LunarLander-v3"
        ),
        "model_filename": model_filename,
        "reward_version": reward_version,
        "reward_config_filename": (
            "reward_config.json"
        ),
        "reward_wrapper_filename": (
            "flip_landing_reward_wrapper.py"
        ),
        "observation_dimensions": 11,
        "architecture": {
            "actor_layers": (
                training_config.get(
                    "actor_layers"
                )
            ),
            "critic_layers": (
                training_config.get(
                    "critic_layers"
                )
            ),
        },
        "evaluation": summary,
    }

    (
        staging_directory
        / "config.json"
    ).write_text(
        json.dumps(
            config_payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    # ------------------------------------------------------------------
    # Model-card content
    # ------------------------------------------------------------------

    actor_layers = training_config.get(
        "actor_layers",
        "Not supplied",
    )

    critic_layers = training_config.get(
        "critic_layers",
        "Not supplied",
    )

    change_notes_markdown = (
        "\n".join(
            f"{index}. {note}"
            for index, note
            in enumerate(
                notes,
                start=1,
            )
        )
        if notes
        else "No change notes supplied."
    )

    reward_table = _markdown_table(
        reward_config
    )

    # Hugging Face reads this object as the YAML
    # metadata at the top of README.md.
    card_data = ModelCardData(
        library_name=(
            "stable-baselines3"
        ),
        pipeline_tag=(
            "reinforcement-learning"
        ),
        tags=[
            "stable-baselines3",
            "reinforcement-learning",
            "deep-reinforcement-learning",
            "ppo",
            "LunarLander-v3",
            "custom-reward",
            "reward-shaping",
            "actor-critic",
        ],
    )

    # The placeholders are replaced after dedenting.
    # This avoids multiline tables preventing
    # textwrap.dedent() from working correctly.
    body_template = textwrap.dedent(
        f"""\
        # PPO LunarLander flip, recover and land agent

        This repository contains a Stable-Baselines3 PPO
        actor-critic agent trained on a customised
        `LunarLander-v3` environment.

        ## Learned task

        The curriculum teaches one policy to:

        1. complete a full rotation in a fixed direction;
        2. recover upright and arrest angular motion;
        3. enter the landing zone;
        4. reduce descent speed and land safely.

        Reward configuration version: `{reward_version}`.

        ## Changes in this upload

        __CHANGE_NOTES__

        ## Reward design

        The shaped reward includes:

        - one-off rotation-progress and flip-completion rewards;
        - an upright post-flip recovery reward;
        - horizontal guidance towards the landing zone;
        - an altitude-dependent vertical-speed target;
        - attitude and angular-speed control;
        - a near-ground descent-overspeed penalty;
        - distinct penalties for off-zone landings and in-zone crashes.

        __REWARD_TABLE__

        ## Evaluation

        Deterministic evaluation over
        {summary['n_episodes']} fixed-seed episodes:

        | Metric | Value |
        |---|---:|
        | Mean shaped reward | {summary['mean_shaped_reward']:.2f} |
        | Mean original reward | {summary['mean_original_reward']:.2f} |
        | Full-rotation rate | {summary['flip_completion_rate']:.1%} |
        | Recovery rate | {summary['recovery_completion_rate']:.1%} |
        | Recovery given a flip | {summary['recovery_given_flip_rate']:.1%} |
        | Safe-landing rate | {summary['safe_landing_rate']:.1%} |
        | Flip-and-land rate | {summary['flip_landing_rate']:.1%} |
        | Terminal in-zone rate | {summary.get('inside_landing_zone_rate', 0.0):.1%} |
        | In-zone crash rate | {summary.get('in_zone_crash_rate', 0.0):.1%} |

        ## Architecture

        - Algorithm: PPO
        - Policy: MLP actor-critic
        - Actor hidden layers: `{actor_layers}`
        - Critic hidden layers: `{critic_layers}`
        - Observation dimensions: `11`
        - Discrete actions: `4`

        ## Training configuration

        | Parameter | Value |
        |---|---:|
        | Phase timesteps | {training_config.get('total_timesteps')} |
        | Parallel environments | {training_config.get('n_envs')} |
        | Learning rate | {training_config.get('learning_rate')} |
        | Rollout steps per environment | {training_config.get('n_steps')} |
        | Batch size | {training_config.get('batch_size')} |
        | Optimisation epochs | {training_config.get('n_epochs')} |
        | Gamma | {training_config.get('gamma')} |
        | GAE lambda | {training_config.get('gae_lambda')} |
        | Entropy coefficient | {training_config.get('ent_coef')} |
        | PPO clip range | {training_config.get('clip_range')} |
        | Training seed | {training_config.get('seed')} |

        ## Replay

        - Seed: `{replay_details['seed']}`
        - Original reward: `{replay_details['original_reward']:.2f}`
        - Shaped reward: `{replay_details['shaped_reward']:.2f}`
        - Rotations completed: `{replay_details['rotations_completed']:.2f}`
        - Flip completed: `{replay_details['flip_completed']}`
        - Recovery completed:
          `{replay_details.get('recovery_completed', 'Not recorded')}`
        - Landed safely: `{replay_details['landed_safely']}`
        - Outcome: `{replay_details.get('custom_outcome', 'Not recorded')}`

        [![Flip, recovery and landing replay](replay.gif)](replay.mp4)

        [Open the full MP4 replay](replay.mp4)

        ## Repository files

        - `{model_filename}`: selected PPO model
        - `flip_landing_reward_wrapper.py`: custom environment wrapper
        - `training_config.json`: PPO training settings
        - `reward_config.json`: reward configuration
        - `episode_results.csv`: fixed-seed evaluation episodes
        - `results.json`: machine-readable evaluation summary
        - `config.json`: compact model metadata
        - `replay.gif`: model-card preview
        - `replay.mp4`: full replay
        """
    ).strip()

    # These values contain multiple lines, so they are
    # inserted only after the main template is dedented.
    body = (
        body_template
        .replace(
            "__CHANGE_NOTES__",
            change_notes_markdown,
        )
        .replace(
            "__REWARD_TABLE__",
            reward_table,
        )
    )

    # The YAML header must begin at column zero at the
    # very start of README.md.
    readme = (
        "---\n"
        f"{card_data.to_yaml()}\n"
        "---\n\n"
        f"{body}\n"
    )

    # Parse the card locally before publishing it.
    model_card = ModelCard(readme)

    parsed_metadata = (
        model_card.data.to_dict()
    )

    assert (
        parsed_metadata["library_name"]
        == "stable-baselines3"
    )

    assert (
        parsed_metadata["pipeline_tag"]
        == "reinforcement-learning"
    )

    readme_path = (
        staging_directory
        / "README.md"
    )

    readme_path.write_text(
        model_card.content,
        encoding="utf-8",
    )

    # ------------------------------------------------------------------
    # Upload and verification
    # ------------------------------------------------------------------

    print("Prepared files:")

    for path in sorted(
        staging_directory.iterdir()
    ):
        print(
            "-",
            path.name,
            (
                f"({path.stat().st_size / 1024:.1f} "
                "KiB)"
            ),
        )

    api = HfApi()

    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        private=private,
        exist_ok=True,
    )

    upload_result = api.upload_folder(
        folder_path=str(
            staging_directory
        ),
        repo_id=repo_id,
        repo_type="model",
        commit_message=commit_message,
    )

    uploaded_files = set(
        api.list_repo_files(
            repo_id=repo_id,
            repo_type="model",
        )
    )

    required_uploaded_files = {
        "README.md",
        model_filename,
        "replay.gif",
        "replay.mp4",
        "reward_config.json",
        "training_config.json",
        "episode_results.csv",
        "results.json",
        "config.json",
        "flip_landing_reward_wrapper.py",
    }

    missing_files = (
        required_uploaded_files
        - uploaded_files
    )

    if missing_files:
        raise RuntimeError(
            "Upload completed but these files "
            f"are missing: {sorted(missing_files)}"
        )

    print()
    print(
        "Upload complete:",
        upload_result,
    )
    print(
        "Verified repository files:",
        sorted(
            required_uploaded_files
        ),
    )

    return {
        "repo_id": repo_id,
        "staging_directory": (
            staging_directory
        ),
        "model_card_path": (
            readme_path
        ),
        "summary": summary,
        "reward_version": reward_version,
        "upload_result": upload_result,
    }

## 4. HuggingFace authentication

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## How to run the curriculum

### Full training run

Run Sections 1–4, followed by Phases A, B, C and D in order.

### Continue from Phase C and train only Phase D

Run:

1. Runtime setup
2. Custom environment
3. Shared training infrastructure
4. Hugging Face authentication
5. Phase D only
6. Final evaluation and publication

Do not rerun Phases A–C. The selected Phase C checkpoint already
contains the policy updates learned during all three earlier phases.

## 5. Model training

### 5.1 Phase A - Flip acquisition

In [ ]:
phase_a_hub_checkpoint_callback = (
    HubCheckpointCallback(
        repo_id=HF_REPO_ID,
        phase_name="phase_a_flip_acquisition",
        every_timesteps=1_000_000,
    )
)

In [ ]:
phase_a_run = train_ppo_experiment(
    experiment_name=(
        "ppo_lunarlander_flip_phase_a"
    ),
    total_timesteps=5_000_000,

    env_factory=(
        make_flip_acquisition_lander
    ),

    n_envs=16,
    seed=42,

    actor_layers=(128, 128),
    critic_layers=(128, 128),

    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.03,
    clip_range=0.2,

    eval_every_timesteps=50_000,
    checkpoint_every_timesteps=100_000,
    n_eval_episodes=20,

    extra_callbacks=[
        phase_a_hub_checkpoint_callback,
    ],

    output_root=OUTPUT_ROOT,

    device="cpu",
)

In [ ]:
phase_a_results = (
    evaluate_phase_candidates(
        phase_a_run,
        reward_config=(
            flip_acquisition_config
        ),
        n_episodes=20,
        starting_seed=(
            EVALUATION_START_SEED
        ),
    )
)


phase_a_results = (
    phase_a_results
    .sort_values(
        [
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "mean_shaped_reward",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    phase_a_results[
        [
            "model_path",
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "safe_landing_rate",
            "mean_shaped_reward",
        ]
    ]
)

selected_phase_a_model_path = Path(
    phase_a_results.loc[
        0,
        "model_path",
    ]
)

phase_a_flip_rate = float(
    phase_a_results.loc[
        0,
        "flip_completion_rate",
    ]
)

print(
    "Selected Phase A model:",
    selected_phase_a_model_path,
)

print(
    "Phase A flip rate:",
    f"{phase_a_flip_rate:.1%}",
)

if phase_a_flip_rate == 0.0:
    raise RuntimeError(
        "No Phase A checkpoint completed a flip. "
        "Do not start Phase B. Run another Phase A "
        "experiment with a different seed."
    )

if phase_a_flip_rate < 0.30:
    print(
        "Warning: the best Phase A flip rate is "
        "below 30%. Another Phase A seed may provide "
        "a stronger starting checkpoint."
    )

In [ ]:
upload_selected_phase_artifacts(
    phase_name="phase_a",
    model_path=(
        selected_phase_a_model_path
    ),
    run=phase_a_run,
    reward_config=(
        flip_acquisition_config
    ),
    candidate_results=(
        phase_a_results
    ),
    repo_id=HF_REPO_ID,
)

### 5.2 Phase B - Recovery and landing

In [ ]:
phase_b_hub_checkpoint_callback = (
    HubCheckpointCallback(
        repo_id=HF_REPO_ID,
        phase_name="phase_b_recovery_and_landing",
        every_timesteps=1_000_000,
    )
)

In [ ]:
phase_b_run = train_ppo_experiment(
    experiment_name=(
        "ppo_lunarlander_flip_phase_b"
    ),
    total_timesteps=5_000_000,

    env_factory=(
        make_flip_landing_lander
    ),

    initial_model_path=(
        selected_phase_a_model_path
    ),

    n_envs=16,
    seed=42,

    # These document the loaded architecture.
    actor_layers=(128, 128),
    critic_layers=(128, 128),

    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.03,
    clip_range=0.2,

    eval_every_timesteps=50_000,
    checkpoint_every_timesteps=100_000,
    n_eval_episodes=20,

    extra_callbacks=[
        phase_b_hub_checkpoint_callback
    ],

    output_root=OUTPUT_ROOT,

    device="cpu",
)

In [ ]:
phase_b_results = (
    evaluate_phase_candidates(
        phase_b_run,
        reward_config=(
            flip_landing_config
        ),
        n_episodes=20,
        starting_seed=(
            EVALUATION_START_SEED
        ),
    )
)

phase_b_results = (
    phase_b_results
    .sort_values(
        [
            "flip_landing_rate",
            "landing_given_flip_rate",
            "recovery_given_flip_rate",
            "flip_completion_rate",
            "mean_original_reward",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    phase_b_results[
        [
            "model_path",
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "flip_landing_rate",
            "landing_given_flip_rate",
            "mean_original_reward",
        ]
    ]
)

selected_phase_b_model_path = Path(
    phase_b_results.loc[
        0,
        "model_path",
    ]
)

print(
    "Selected Phase B model:",
    selected_phase_b_model_path,
)

phase_b_evaluation = evaluate_flip_model(
    selected_phase_b_model_path,
    reward_config=flip_landing_config,
    n_episodes=100,
    starting_seed=EVALUATION_START_SEED,
)

display(
    pd.Series(
        phase_b_evaluation["summary"],
        name="Final flip-and-land model",
    )
)

display(
    phase_b_evaluation["episodes"]
    .sort_values(
        [
            "flip_and_landed",
            "recovery_completed",
            "flip_completed",
            "original_reward",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .head(20)
)

In [ ]:
upload_selected_phase_artifacts(
    phase_name="phase_b",
    model_path=(
        selected_phase_b_model_path
    ),
    run=phase_b_run,
    reward_config=(
        flip_landing_config
    ),
    candidate_results=(
        phase_b_results
    ),
    repo_id=HF_REPO_ID,
)

### 5.3 Phase C - Landing-zone acquisition

In [ ]:
phase_c_hub_checkpoint_callback = (
    HubCheckpointCallback(
        repo_id=HF_REPO_ID,
        phase_name="phase_c_landing_zone",
        every_timesteps=500_000,
    )
)

In [ ]:
phase_c_run = train_ppo_experiment(
    experiment_name=(
        "ppo_lunarlander_"
        "flip_phase_c_in_zone"
    ),

    total_timesteps=5_000_000,

    env_factory=(
        make_flip_in_zone_lander
    ),

    initial_model_path=(
        selected_phase_b_model_path
    ),

    n_envs=16,
    seed=42,

    actor_layers=(128, 128),
    critic_layers=(128, 128),

    # Smaller updates reduce the risk of destroying
    # the already learned flip policy.
    learning_rate=1e-4,

    n_steps=1024,
    batch_size=64,
    n_epochs=4,

    gamma=0.999,
    gae_lambda=0.98,

    # Retain some exploration, but less than during
    # initial flip acquisition.
    ent_coef=0.01,

    clip_range=0.15,

    eval_every_timesteps=50_000,
    checkpoint_every_timesteps=100_000,
    n_eval_episodes=20,

    extra_callbacks=[
        phase_c_hub_checkpoint_callback,
    ],

    output_root=OUTPUT_ROOT,

    device="cpu",
)

In [ ]:
phase_c_results = (
    evaluate_phase_candidates(
        phase_c_run,
        reward_config=(
            flip_in_zone_config
        ),
        n_episodes=20,
        starting_seed=(
            EVALUATION_START_SEED
        ),
    )
)


phase_c_results = (
    phase_c_results
    .sort_values(
        [
            "flip_landing_rate",
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "inside_landing_zone_rate",
            "outside_zone_landing_rate",
            "mean_original_reward",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    phase_c_results[
        [
            "model_path",
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "stable_landing_rate",
            "inside_landing_zone_rate",
            "outside_zone_landing_rate",
            "flip_landing_rate",
            "mean_original_reward",
        ]
    ]
)

In [ ]:
from huggingface_hub import HfApi

selected_phase_c_model_path = Path(
    phase_c_results.loc[
        0,
        "model_path",
    ]
)

print(
    "Selected Phase C model:",
    selected_phase_c_model_path,
)

display(
    phase_c_results[
        [
            "model_path",
            "flip_landing_rate",
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "inside_landing_zone_rate",
            "outside_zone_landing_rate",
            "mean_original_reward",
        ]
    ].head(10)
)

upload_selected_phase_artifacts(
    phase_name="phase_c",
    model_path=(
        selected_phase_c_model_path
    ),
    run=phase_c_run,
    reward_config=(
        flip_in_zone_config
    ),
    candidate_results=(
        phase_c_results
    ),
    repo_id=HF_REPO_ID,
)

In [ ]:
phase_c_evaluation = evaluate_flip_model(
    selected_phase_c_model_path,
    reward_config=(
        flip_in_zone_config
    ),
    n_episodes=100,
    starting_seed=EVALUATION_START_SEED
)

display(
    pd.Series(
        phase_c_evaluation["summary"],
        name=(
            "Final Phase C "
            "flip-and-land model"
        ),
    )
)

### 5.4 Phase D - Soft touchdown

#### Download Phase C model

In [ ]:
from huggingface_hub import (
    hf_hub_download,
)

from pathlib import Path

resume_model_path = Path(
    hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=(
            "selected-models/"
            "phase_c_best.zip"
        ),
        repo_type="model",
    )
)

print(
    "Resuming from:",
    resume_model_path,
)

In [ ]:
resume_model = PPO.load(
    resume_model_path,
    device="cpu",
)

phase_d_test_env = (
    make_flip_soft_landing_lander()
)

try:
    print(
        "Saved observation space:",
        resume_model.observation_space,
    )

    print(
        "Phase D observation space:",
        phase_d_test_env.observation_space,
    )

    print(
        "Saved action space:",
        resume_model.action_space,
    )

    print(
        "Phase D action space:",
        phase_d_test_env.action_space,
    )

    assert (
        resume_model.observation_space
        == phase_d_test_env.observation_space
    )

    assert (
        resume_model.action_space
        == phase_d_test_env.action_space
    )

finally:
    phase_d_test_env.close()

print(
    "Phase C checkpoint is compatible "
    "with the Phase D environment."
)

In [ ]:
phase_d_hub_checkpoint_callback = (
    HubCheckpointCallback(
        repo_id=HF_REPO_ID,
        phase_name=(
            "phase_d_soft_touchdown"
        ),
        every_timesteps=500_000,
    )
)

phase_d_run = train_ppo_experiment(
    experiment_name=(
        "ppo_lunarlander_"
        "flip_phase_d_soft_touchdown_01"
    ),

    total_timesteps=2_000_000,

    env_factory=(
        make_flip_soft_landing_lander
    ),

    initial_model_path=(
        resume_model_path
    ),

    n_envs=16,
    seed=43,

    actor_layers=(128, 128),
    critic_layers=(128, 128),

    learning_rate=5e-5,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,

    gamma=0.999,
    gae_lambda=0.98,

    ent_coef=0.005,
    clip_range=0.10,

    eval_every_timesteps=50_000,
    checkpoint_every_timesteps=100_000,
    n_eval_episodes=30,

    extra_callbacks=[
        phase_d_hub_checkpoint_callback,
    ],

    output_root=OUTPUT_ROOT,

    device="cpu",
)

#### Evaluate model Phase D

In [ ]:
phase_d_results = evaluate_phase_candidates(
    phase_d_run,
    reward_config=(
        flip_soft_landing_config
    ),
    n_episodes=30,
    starting_seed=EVALUATION_START_SEED,
)

phase_d_results = (
    phase_d_results
    .sort_values(
        [
            "flip_landing_rate",
            "landing_given_flip_rate",
            "recovery_given_flip_rate",
            "flip_completion_rate",
            "in_zone_crash_rate",
            "mean_original_reward",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    phase_d_results[
        [
            "model_path",
            "flip_landing_rate",
            "safe_landing_rate",
            "flip_completion_rate",
            "recovery_given_flip_rate",
            "inside_landing_zone_rate",
            "in_zone_crash_rate",
            "mean_original_reward",
        ]
    ].head(15)
)


selected_phase_d_model_path = Path(
    phase_d_results.loc[
        0,
        "model_path",
    ]
)

print(
    "Selected Phase D model:",
    selected_phase_d_model_path,
)

upload_selected_phase_artifacts(
    phase_name="phase_d",
    model_path=(
        selected_phase_d_model_path
    ),
    run=phase_d_run,
    reward_config=(
        flip_soft_landing_config
    ),
    candidate_results=(
        phase_d_results
    ),
    repo_id=HF_REPO_ID,
)

phase_d_evaluation = evaluate_flip_model(
    selected_phase_d_model_path,
    reward_config=(
        flip_soft_landing_config
    ),
    n_episodes=100,
    starting_seed=EVALUATION_START_SEED,
)

display(
    pd.Series(
        phase_d_evaluation["summary"],
        name=(
            "Final Phase D "
            "soft-touchdown model"
        ),
    )
)

## 6. Final evaluation and publication

In [ ]:
episode_results = (
    phase_d_evaluation["episodes"]
)

successful_flip_landings = (
    episode_results[
        episode_results[
            "flip_and_landed"
        ]
    ]
    .copy()
)

if successful_flip_landings.empty:
    video_seed = int(
        episode_results
        .sort_values(
            "shaped_reward",
            ascending=False,
        )
        .iloc[0]["seed"]
    )

    print(
        "No successful flip-and-land episode "
        "was found in the evaluation sample."
    )
    print(
        "Using the highest shaped-reward "
        "episode instead."
    )

else:
    video_seed = int(
        successful_flip_landings
        .sort_values(
            "original_reward",
            ascending=False,
        )
        .iloc[0]["seed"]
    )

    print(
        "Selected the successful flip-and-land "
        "episode with the highest original reward."
    )

print("Selected video seed:", video_seed)

In [ ]:
phase_d_replay = record_flip_replay(
    selected_phase_d_model_path,
    output_path=(
        "/content/"
        "LunarLander-v3_Phase-D_Replay.mp4"
    ),
    seed=video_seed,
    reward_config=(
        flip_soft_landing_config
    ),
)

In [ ]:
from IPython.display import (
    Video,
    display,
)


display(
    Video(
        phase_d_replay["path"],
        embed=True,
    )
)

In [ ]:
upload_result = upload_flip_model_to_hub(
    selected_phase_d_model_path,

    repo_id=HF_REPO_ID,

    training_config_path=(
        phase_d_run["config_path"]
    ),

    reward_wrapper_path=(
        REWARD_WRAPPER_PATH
    ),

    reward_config=(
        flip_soft_landing_config
    ),

    evaluation=(
        phase_d_evaluation
    ),

    replay=(
        phase_d_replay
    ),

    reward_version=(
        "v4-soft-touchdown"
    ),

    change_notes=[
        (
            "Continued from the selected "
            "Phase C checkpoint."
        ),
        (
            "Added an altitude-dependent "
            "vertical-speed target."
        ),
        (
            "Added a near-ground quadratic "
            "descent-overspeed penalty."
        ),
        (
            "Added a dedicated in-zone "
            "crash penalty."
        ),
        (
            "Reduced risky late horizontal "
            "corrections using a deadband."
        ),
    ],
)

## Appendix: Colab troubleshooting

In [ ]:
# Clean progress display
from rich import get_console

console = get_console()

# Rich versions that store a single active display.
active_live = getattr(console, "_live", None)

if active_live is not None:
    try:
        active_live.stop()
    except Exception:
        try:
            console.clear_live()
        except Exception:
            pass

# Newer Rich versions that store a stack of displays.
live_stack = getattr(console, "_live_stack", None)

if live_stack:
    while live_stack:
        active_live = live_stack[-1]

        try:
            active_live.stop()
        except Exception:
            try:
                console.clear_live()
            except Exception:
                break

print("Rich live display cleared.")